# Whisper + pyannote + Sortformer — one environment, plus a noise sweep

Two things in one notebook, in the order they have to happen:

**Part A — the environment test.** Does NeMo (which Sortformer comes from) actually
coexist with Whisper and pyannote in a single Python environment? This is the question
Dr. Ayyaz asked. It is an install test with a clear pass or fail, not a modelling
exercise.

**Part B — Sortformer under noise.** The one measurement the August benchmark never
made, and the one your own report listed as the most useful thing left to do.

### How to run it

1. Runtime → Change runtime type → **T4 GPU**.
2. Run Part A cell 1. It will take 15–30 minutes, mostly building NeMo.
3. **Restart the runtime when it tells you to.** This is not optional — numpy is
   downgraded, and anything imported before the restart keeps the old copy.
4. Run everything after that in order.

### Two things to be careful about

- Use **base.en**, not the `large-v3` the vast.ai tutorial uses. That tutorial assumes a
  48 GB card; you are on a 16 GB T4. Your own measured pipeline peaked at 1.26 GB, so
  size is a non-issue *provided* you do not copy their model choice.
- The tutorial installs NeMo from the live `main` branch with no fixed version, so what
  works today may not work next month. Cell 2 saves a full `pip freeze` for exactly this
  reason. Keep that file — it is the only record of what worked.

If Part A fails after an afternoon, stop. "I tried it in a clean environment and here is
the exact error" is a legitimate result to send him, and it is worth more than a vague
"it didn't work".

## Part A — environment test

### A1. Install

In [ ]:
# NeMo first. It will drag numpy up to 2.x; that gets fixed at the end.
!pip install -q "nemo_toolkit[asr]"

# The two the benchmark already used.
!pip install -q openai-whisper "pyannote.audio==4.0.7" soundfile

# The actual fix, and it has to come LAST or the installs above undo it.
# NeMo needs numpy below 2.0. With numpy 2.x it falls back to a broken helper
# package, which is the error you just saw (NVIDIA issue #15331).
!pip uninstall -y numba-cuda
!pip install -q "numpy==1.26.4" --force-reinstall --no-deps

import subprocess, sys
out = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
print("\n".join(l for l in out.split("\n") if l.startswith(("numpy", "numba"))))

print("\n" + "=" * 70)
print("Check the line above says numpy 1.26.4, NOT 2.x.")
print("If it does: Runtime -> Restart session, then run A2.")
print("=" * 70)

Found existing installation: numba-cuda 0.22.2
Uninstalling numba-cuda-0.22.2:
  Successfully uninstalled numba-cuda-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.2 MB/s eta 0:00:00
numba                                       0.67.0
numpy                                       1.26.4

Check the line above says numpy 1.26.4, NOT 2.x.
If it does: Runtime -> Restart session, then run A2.


### A2. Does everything import together?

This is the actual test. If all three import in one process and report a version, the
environment question is answered.

In [ ]:
import sys, subprocess

results = {}

def check(label, fn):
    try:
        results[label] = fn()
        print(f"  OK    {label}: {results[label]}")
    except Exception as e:
        results[label] = None
        print(f"  FAIL  {label}: {type(e).__name__}: {e}")

print("Importing all three in one process:\n")

check("numpy",      lambda: __import__("numpy").__version__)
check("torch",      lambda: __import__("torch").__version__)
check("whisper",    lambda: (__import__("whisper"), "imported")[1])
check("pyannote",   lambda: __import__("pyannote.audio", fromlist=["__version__"]).__version__)
check("nemo asr",   lambda: (__import__("nemo.collections.asr", fromlist=["models"]), "imported")[1])

import torch
print(f"\n  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to T4'}")

# The record of what actually worked. Download this and keep it.
with open("pip_freeze_working.txt", "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                           capture_output=True, text=True).stdout)

ok = all(v is not None for v in results.values())
print("\n" + "=" * 70)
print("PASS - all three coexist in one environment." if ok else
      "FAIL - see the errors above. This is a reportable result, not a dead end.")
print("=" * 70)

Importing all three in one process:

  OK    numpy: 1.26.4
  OK    torch: 2.11.0+cu128
  OK    whisper: imported
  OK    pyannote: 4.0.7


  OK    nemo asr: imported

  GPU: Tesla T4

PASS - all three coexist in one environment.


### A3. The project's audio and its scoring code

Cloning the repo rather than uploading files by hand keeps the reference transcripts and
the metric functions identical to the ones behind every earlier figure. That is what
makes the numbers comparable.

In [ ]:
import os, sys, glob, inspect

if not os.path.exists("emr-assistant-backend"):
    !git clone -q https://github.com/Wajeeha-Kamran/emr-assistant-backend.git
sys.path.insert(0, "/content/emr-assistant-backend")

AUDIO_DIR = "/content/emr-assistant-backend/docs/evidence/human_distinct"
AUDIO = sorted(glob.glob(os.path.join(AUDIO_DIR, "consult_*.wav")))
print(f"Audio files: {len(AUDIO)}")
for p in AUDIO:
    print("   ", os.path.basename(p))

# These load without a database or a .env because the app imports inside
# evaluate_accuracy.py sit inside main().
from scripts.evaluate_accuracy import word_error_rate, speaker_accuracy, parse_scripts, normalise

# Signatures printed on purpose: if any of these differ from what the scoring cell
# assumes, you find out here rather than twenty minutes into a run.
print("\nScoring functions as they are actually defined:")
for fn in (word_error_rate, speaker_accuracy, parse_scripts, normalise):
    print(f"   {fn.__name__}{inspect.signature(fn)}")

Audio files: 4
    consult_1.wav
    consult_2.wav
    consult_3.wav
    consult_4.wav

Scoring functions as they are actually defined:
   word_error_rate(ref: List[str], hyp: List[str]) -> float
   speaker_accuracy(ref_words: List[str], ref_spk: List[str], hyp_words: List[str], hyp_spk: List[str]) -> Tuple[int, int]
   parse_scripts(path: str) -> Dict[int, List[Tuple[str, str]]]
   normalise(text: str) -> List[str]


### A4. Load the two models, once

In [ ]:
import time, torch, whisper
from nemo.collections.asr.models import SortformerEncLabelModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

t = time.time()
asr = whisper.load_model("base.en", device=DEVICE)
print(f"Whisper base.en loaded in {time.time() - t:.1f}s")

t = time.time()
diarizer = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
diarizer.eval()
if DEVICE == "cuda":
    diarizer = diarizer.cuda()
print(f"Sortformer loaded in {time.time() - t:.1f}s")

print(f"\nPeak VRAM after loading both: {torch.cuda.max_memory_allocated()/1e9:.2f} GB"
      if DEVICE == "cuda" else "")
print("Both models are live in the same process. That is the environment question answered.")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 83.9MiB/s]


Whisper base.en loaded in 4.3s


diar_sortformer_4spk-v1.nemo: reconstructing file:   0%|          |  0.00B /  493MB            

diar_sortformer_4spk-v1.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-08-21 06:59:49 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-21 06:59:49 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-21 06:59:51 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
Sortformer loaded in 29.4s

Peak VRAM after loading both: 1.30 GB
Both models are live in the same process. That is the environment question answered.


### A5. The pipeline, on one file

**Read the raw Sortformer output this cell prints before trusting anything after it.**
NeMo's return shape is the single most likely thing to have changed between versions, and
the parser below makes an assumption about it.

If you still have your PASTE_ cells from the 19 August run, the parsing code in them is
already known-good against your data — prefer it over the parser here.

In [ ]:
raw = diarizer.diarize(audio=[AUDIO[0]], batch_size=1)

print("Raw Sortformer output, so you can see its actual shape:\n")
print("  type:", type(raw))
print("  len :", len(raw))
first = raw[0]
print("  [0] type:", type(first), "len:", len(first) if hasattr(first, "__len__") else "n/a")
print("\n  first few entries:")
for entry in (first[:5] if isinstance(first, (list, tuple)) else [first]):
    print("   ", repr(entry))

[NeMo I 2026-08-21 07:04:41 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-08-21 07:04:42 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:02,  2.31s/it]

Raw Sortformer output, so you can see its actual shape:

  type: <class 'list'>
  len : 1
  [0] type: <class 'list'> len: 22

  first few entries:
    '1.120 4.960 speaker_0'
    '12.480 14.320 speaker_0'
    '24.160 26.880 speaker_0'
    '33.920 37.520 speaker_0'
    '43.520 45.760 speaker_0'


In [ ]:
def parse_sortformer(raw_one):
    """Turn Sortformer's output into [(start, end, speaker_label), ...].

    Expects entries like "0.00 3.52 speaker_0". If the cell above showed a different
    shape, this is the function to adjust - nothing else downstream cares how the spans
    were produced.
    """
    spans = []
    for entry in raw_one:
        if isinstance(entry, str):
            parts = entry.split()
            if len(parts) >= 3:
                spans.append((float(parts[0]), float(parts[1]), parts[2]))
        elif isinstance(entry, (list, tuple)) and len(entry) >= 3:
            spans.append((float(entry[0]), float(entry[1]), str(entry[2])))
    return spans


def words_with_speakers(audio_path):
    """Whisper words + Sortformer spans -> turns, in one process."""
    result = asr.transcribe(audio_path, language="en", word_timestamps=True, verbose=False)

    words = []
    for seg in result["segments"]:
        for w in seg.get("words", []):
            words.append({"text": w["word"].strip(),
                          "start": w["start"], "end": w["end"]})

    spans = parse_sortformer(diarizer.diarize(audio=[audio_path], batch_size=1)[0])

    # Each word goes to whichever speaker span it overlaps most.
    for w in words:
        best, best_overlap = None, 0.0
        for s, e, spk in spans:
            overlap = min(w["end"], e) - max(w["start"], s)
            if overlap > best_overlap:
                best, best_overlap = spk, overlap
        w["speaker"] = best or (spans[0][2] if spans else "speaker_0")

    # Consecutive words by the same speaker become one turn.
    turns = []
    for w in words:
        if turns and turns[-1]["speaker"] == w["speaker"]:
            turns[-1]["text"] += " " + w["text"]
        else:
            turns.append({"speaker": w["speaker"], "text": w["text"]})
    return turns


def assign_roles(turns):
    """Whoever asks more questions is the doctor.

    Identical to DiarizationService, and held constant across every run in the August
    benchmark. Changing it here would turn a model comparison into a comparison of
    naming rules.
    """
    questions = {}
    for t in turns:
        questions[t["speaker"]] = questions.get(t["speaker"], 0) + t["text"].count("?")

    if questions and max(questions.values()) > 0:
        doctor = max(questions, key=questions.get)
    else:
        doctor = turns[0]["speaker"] if turns else None

    return [{"speaker_role": "DOCTOR" if t["speaker"] == doctor else "PATIENT",
             "text": t["text"]} for t in turns]


smoke = assign_roles(words_with_speakers(AUDIO[0]))
print(f"{len(smoke)} turns from {os.path.basename(AUDIO[0])}:\n")
for t in smoke[:6]:
    print(f"  {t['speaker_role']:8} {t['text'][:70]}")

100%|██████████| 9503/9503 [00:07<00:00, 1195.19frames/s]

[NeMo I 2026-08-21 07:05:07 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



[NeMo W 2026-08-21 07:05:07 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:00,  1.65it/s]

14 turns from consult_1.wav:

  DOCTOR   Good morning. Please take a seat. What brings you in today?
  PATIENT  I have been having really bad adage for about 4 days now.
  DOCTOR   Can you describe the pain for me?
  PATIENT  It's mostly on the right side, behind my eye. It drops bright light ma
  DOCTOR   Have you felt sick at all or been vomiting? I
  PATIENT  know she has yesterday morning, but I have actually been sick.


### A6. Clean audio — does it reproduce the benchmark?

The August figures for base.en + Sortformer were **88.3% word / 99.9% speaker**, with
per-script word accuracy of 86.9 / 94.8 / 87.8 / 83.6.

Whisper is deterministic, so the word accuracy should come back *identical*. If it does,
the environment is behaving and everything after this is trustworthy. If it does not,
stop and find out why before running the noise sweep.

In [ ]:
import pandas as pd, re

SCRIPTS = "/content/emr-assistant-backend/docs/evidence/consultation_scripts.md"
references = parse_scripts(SCRIPTS)
print("Scripts in the answer key:", sorted(references.keys()))

def script_number(path):
    """consult_3.wav -> 3, and snr10_consult_3.wav -> 3 as well.
    Matching on 'consult_' matters: a plain digit search would grab the 10
    from the noise level and look up the wrong script."""
    return int(re.search(r"consult_(\d+)", os.path.basename(path)).group(1))

def flatten(turns):
    """[(speaker, text), ...] -> every word, and who said each one.
    Two parallel lists, which is the shape evaluate_accuracy.py works in."""
    words, speakers = [], []
    for speaker, text in turns:
        w = normalise(text)
        words.extend(w)
        speakers.extend([speaker] * len(w))
    return words, speakers

def score(hyp_turns, ref_turns):
    ref_words, ref_spk = flatten(ref_turns)
    hyp_words, hyp_spk = flatten([(t["speaker_role"], t["text"]) for t in hyp_turns])

    word_acc = (1 - word_error_rate(ref_words, hyp_words)) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    speaker_acc = (correct / total * 100) if total else 0.0
    return word_acc, speaker_acc

rows = []
for path in AUDIO:
    n = script_number(path)
    t0 = time.time()
    turns = assign_roles(words_with_speakers(path))
    elapsed = time.time() - t0
    word_acc, spk_acc = score(turns, references[n])
    rows.append({"script": n, "word_acc_%": round(word_acc, 1),
                 "speaker_acc_%": round(spk_acc, 1), "seconds": round(elapsed, 1)})
    print(f"  script {n}: {word_acc:.1f}% words, {spk_acc:.1f}% speakers, {elapsed:.1f}s")

clean = pd.DataFrame(rows).sort_values("script")
print("\n", clean.to_string(index=False))
print(f"\n  MEAN: {clean['word_acc_%'].mean():.1f}% word / {clean['speaker_acc_%'].mean():.1f}% speaker")
print("  August benchmark: 88.3% word / 99.9% speaker")
print(f"\n  Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB (benchmark 1.26 GB)")

Scripts in the answer key: [1, 2, 3, 4]


100%|██████████| 9503/9503 [00:02<00:00, 3304.28frames/s]

[NeMo I 2026-08-21 07:11:03 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



[NeMo W 2026-08-21 07:11:03 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:00,  1.88it/s]


  script 1: 86.9% words, 100.0% speakers, 3.6s


...

  script 3: 87.8% words, 99.4% speakers, 3.9s


100%|██████████| 13734/13734 [00:04<00:00, 3170.05frames/s]

[NeMo I 2026-08-21 07:11:16 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



[NeMo W 2026-08-21 07:11:16 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:00,  1.10it/s]

  script 4: 83.3% words, 99.7% speakers, 5.5s

  script  word_acc_%  speaker_acc_%  seconds
      1        86.9          100.0      3.6
      2        94.4           99.5      3.6
      3        87.8           99.4      3.9
      4        83.3           99.7      5.5

  MEAN: 88.1% word / 99.6% speaker
  August benchmark: 88.3% word / 99.9% speaker

  Peak VRAM: 1.56 GB (benchmark 1.26 GB)


## Part B — Sortformer under noise

The benchmark tested noise against **pyannote** only, and the speaker numbers that came
back were unusable: they went *up* under noise, because noise happened to stop the two
voices merging on script 2. That was a fragile recording flipping from wrong to right,
not robustness, and the report correctly threw the result away.

Sortformer separates all four scripts cleanly, so it has no such failure to accidentally
repair. Whatever comes out of this is a real measurement.

Same three levels and the same fixed seed as the August run, so the word-accuracy numbers
line up with the ones already in the report: clean 88.3 → 20 dB 87.7 → 10 dB 82.9 →
5 dB 78.3.

In [ ]:
import numpy as np, soundfile as sf

SEED = 42
SNRS = [20, 10, 5]

def add_noise(src, dst, snr_db, seed=SEED):
    audio, sr = sf.read(src)
    rng = np.random.default_rng(seed)
    signal_power = np.mean(audio ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noisy = audio + rng.normal(0, np.sqrt(noise_power), audio.shape)
    sf.write(dst, np.clip(noisy, -1.0, 1.0), sr)
    return dst

os.makedirs("/content/noisy", exist_ok=True)
noisy_sets = {}
for snr in SNRS:
    noisy_sets[snr] = [
        add_noise(p, f"/content/noisy/snr{snr}_{os.path.basename(p)}", snr)
        for p in AUDIO
    ]
    print(f"  {snr} dB: {len(noisy_sets[snr])} files written")

print("\nWritten to /content, so docs/evidence is untouched.")

  20 dB: 4 files written
  10 dB: 4 files written
  5 dB: 4 files written

Written to /content, so docs/evidence is untouched.


In [ ]:
all_rows = [{"condition": "clean", "script": r["script"],
             "word_acc_%": r["word_acc_%"], "speaker_acc_%": r["speaker_acc_%"]}
            for r in rows]

for snr in SNRS:
    print(f"\n{snr} dB SNR:")
    for path in noisy_sets[snr]:
        n = script_number(path)
        turns = assign_roles(words_with_speakers(path))
        word_acc, spk_acc = score(turns, references[n])
        all_rows.append({"condition": f"{snr} dB", "script": n,
                         "word_acc_%": round(word_acc, 1),
                         "speaker_acc_%": round(spk_acc, 1)})
        print(f"  script {n}: {word_acc:.1f}% words, {spk_acc:.1f}% speakers")

detail = pd.DataFrame(all_rows)
summary = detail.groupby("condition", sort=False)[["word_acc_%", "speaker_acc_%"]].mean().round(1)

print("\n" + "=" * 62)
print("SORTFORMER UNDER NOISE")
print("=" * 62)
print(summary.to_string())
print("""
pyannote on the same audio, for comparison:
  word     88.3 -> 87.7 (20 dB) -> 82.9 (10 dB) -> 78.3 (5 dB)
  speaker  unusable - see section 5 of the comparison report
""")
print("Speaker accuracy per script (means hide single-recording collapses):")
print(detail.pivot(index="script", columns="condition", values="speaker_acc_%").to_string())

detail.to_csv("sortformer_noise_detail.csv", index=False)
summary.to_csv("sortformer_noise_summary.csv")
print("\nSaved: sortformer_noise_detail.csv, sortformer_noise_summary.csv")


20 dB SNR:


100%|██████████| 9503/9503 [00:02<00:00, 3319.17frames/s]

[NeMo I 2026-08-21 07:17:15 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



[NeMo W 2026-08-21 07:17:15 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:00,  1.88it/s]


  script 1: 87.3% words, 100.0% speakers


...

  script 3: 70.7% words, 99.3% speakers


100%|██████████| 13734/13734 [00:04<00:00, 2828.81frames/s]

[NeMo I 2026-08-21 07:18:01 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



[NeMo W 2026-08-21 07:18:01 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: num_spks,soft_label_thres,session_len_sec
Diarizing: 1it [00:00,  1.12it/s]

  script 4: 79.1% words, 99.6% speakers

SORTFORMER UNDER NOISE
           word_acc_%  speaker_acc_%
condition                           
clean            88.1           99.6
20 dB            87.4           99.6
10 dB            81.9           99.4
5 dB             76.4           99.3

pyannote on the same audio, for comparison:
  word     88.3 -> 87.7 (20 dB) -> 82.9 (10 dB) -> 78.3 (5 dB)
  speaker  unusable - see section 5 of the comparison report

Speaker accuracy per script (means hide single-recording collapses):
condition  10 dB  20 dB  5 dB  clean
script                              
1          100.0  100.0  99.4  100.0
2           98.9   99.5  98.9   99.5
3           99.3   99.4  99.3   99.4
4           99.3   99.7  99.6   99.7

Saved: sortformer_noise_detail.csv, sortformer_noise_summary.csv


## What to send him

If Part A passed and Part B ran, you have two concrete results:

1. **The environment question is settled** — all three in one process, with a `pip freeze`
   recording exactly which versions did it.
2. **Sortformer under noise, measured** — the gap your own report flagged as the most
   useful thing left to do.

If Part A failed, send the error from A2 and the fact that you tried it in a clean
environment as he suggested. That is still an answer, and it is the one he asked for.

Either way, do not quote a mean speaker accuracy without the per-script numbers beside it.
That is the mistake the 77.6% figure taught this project, and it is worth not repeating.